# Trading System — Analysis Notebook
Loads processed data (from parquet if available, else fetches fresh).

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

from utils.helpers import load_config, load_from_parquet, save_to_parquet
from providers.yfinance_provider import YFinanceProvider
from analysis.indicators import Indicators
from analysis.screener import Screener

config   = load_config('../config/settings.yaml')
provider = YFinanceProvider(config)
ind      = Indicators(config)
screener = Screener(config)
print('✅ Setup complete')

In [ ]:
# -------------------------------------------------------
# SETTINGS — change these
# -------------------------------------------------------
SYMBOL   = 'RELIANCE.NS'
INTERVAL = '1d'
PERIOD   = '1y'
# -------------------------------------------------------

def load_data(symbol, interval, period):
    """Load from parquet if available, else fetch fresh."""
    try:
        df = load_from_parquet(symbol, '../data/processed', interval)
        print(f'✅ Loaded from parquet: {symbol} ({interval})')
        return df
    except FileNotFoundError:
        print(f'⚠️  Parquet not found — fetching fresh data for {symbol}')
        df = provider.get_historical_data(symbol, period=period, interval=interval)
        df = ind.add_all(df)
        save_to_parquet(df, symbol, '../data/processed', interval)
        print(f'✅ Fetched and saved: {symbol} ({interval})')
        return df

df = load_data(SYMBOL, INTERVAL, PERIOD)
print(f'Shape: {df.shape}')
df[['Close','RSI','MACD','SMA_50','BB_Upper','BB_Lower']].tail(5)

In [ ]:
# -------------------------------------------------------
# CANDLESTICK CHART with indicators
# -------------------------------------------------------
fig = make_subplots(
    rows=3, cols=1,
    shared_xaxes=True,
    vertical_spacing=0.05,
    row_heights=[0.6, 0.2, 0.2],
    subplot_titles=[f'{SYMBOL} Price', 'RSI', 'MACD']
)

# Candlestick
fig.add_trace(go.Candlestick(
    x=df.index, open=df['Open'], high=df['High'],
    low=df['Low'], close=df['Close'], name='Price'
), row=1, col=1)

# SMAs
for sma, color in [('SMA_20','orange'),('SMA_50','blue'),('SMA_100','red')]:
    if sma in df.columns:
        fig.add_trace(go.Scatter(x=df.index, y=df[sma], name=sma,
            line=dict(color=color, width=1)), row=1, col=1)

# Bollinger Bands
fig.add_trace(go.Scatter(x=df.index, y=df['BB_Upper'], name='BB Upper',
    line=dict(color='gray', dash='dash', width=1)), row=1, col=1)
fig.add_trace(go.Scatter(x=df.index, y=df['BB_Lower'], name='BB Lower',
    line=dict(color='gray', dash='dash', width=1),
    fill='tonexty', fillcolor='rgba(128,128,128,0.05)'), row=1, col=1)

# RSI
fig.add_trace(go.Scatter(x=df.index, y=df['RSI'], name='RSI',
    line=dict(color='purple', width=1.5)), row=2, col=1)
fig.add_hline(y=70, line_dash='dash', line_color='red',   row=2, col=1)
fig.add_hline(y=30, line_dash='dash', line_color='green', row=2, col=1)

# MACD
fig.add_trace(go.Scatter(x=df.index, y=df['MACD'],        name='MACD',
    line=dict(color='blue',  width=1.5)), row=3, col=1)
fig.add_trace(go.Scatter(x=df.index, y=df['MACD_Signal'], name='Signal',
    line=dict(color='orange', width=1.5)), row=3, col=1)
fig.add_bar(x=df.index, y=df['MACD_Hist'], name='Histogram', row=3, col=1)

fig.update_layout(
    height=800, title=f'{SYMBOL} — Technical Analysis ({INTERVAL})',
    xaxis_rangeslider_visible=False, template='plotly_dark'
)
fig.show()

In [ ]:
# -------------------------------------------------------
# SCREENER — ranked table across all symbols
# -------------------------------------------------------
SYMBOLS = [
    'RELIANCE.NS','TCS.NS','HDFCBANK.NS','INFY.NS',
    'ICICIBANK.NS','HINDUNILVR.NS','SBIN.NS','BAJFINANCE.NS',
    'BHARTIARTL.NS','KOTAKBANK.NS','MARUTI.NS','WIPRO.NS','AXISBANK.NS'
]

screener.data_dir = '../data/processed'
screen_df = screener.screen(SYMBOLS, INTERVAL)
screen_df.style.background_gradient(subset=['Composite'], cmap='RdYlGn')

In [ ]:
# -------------------------------------------------------
# COMPARE MULTIPLE STOCKS — normalised price chart
# -------------------------------------------------------
COMPARE = ['RELIANCE.NS', 'TCS.NS', 'INFY.NS', 'HDFCBANK.NS']

fig2 = go.Figure()
for sym in COMPARE:
    try:
        d = load_data(sym, INTERVAL, PERIOD)
        normalised = (d['Close'] / d['Close'].iloc[0]) * 100
        fig2.add_trace(go.Scatter(x=d.index, y=normalised, name=sym))
    except Exception as e:
        print(f'Skipping {sym}: {e}')

fig2.update_layout(
    title='Normalised Price Comparison (Base=100)',
    height=500, template='plotly_dark',
    yaxis_title='Indexed Price'
)
fig2.show()